# Backend query-answer walkthrough

This notebook walks through the backend pipeline cell by cell, using the same services that power the app. The goal is inspection, not performance: each step keeps intermediate objects visible so you can see what the system understood, what it resolved, what evidence it retrieved, what it sent to the answer synthesizer, and what the deterministic coverage layer found.

The notebook intentionally calls a few private helper methods for debugging. That is useful here, but production code should keep using the public service methods and API endpoints.

## 0. Setup

Run this notebook from the project root or directly from `notebooks/`. The setup cell finds the repository root, loads `.env`, and imports the backend services.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repository root.")


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

print(f"Repository root: {ROOT}")

In [ ]:
from src.dossier.builder import DossierBuilder
from src.dossier.openfda_store import OpenFDALabelStore
from src.dossier.rxnorm_store import RxNormParquetStore
from src.query_answer.config import load_query_answer_parameters
from src.query_answer.coverage import (
    add_coverage_limitations,
    build_evidence_coverage,
)
from src.query_answer.models import EvidenceAnswer
from src.query_answer.service import QueryAnswerService
from src.query_answer.synthesizer import EvidenceAnswerSynthesizer
from src.query_understanding.extractor import HybridQueryExtractor
from src.query_understanding.service import QueryUnderstandingService


def to_jsonable(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value


def show_json(value):
    display(JSON(to_jsonable(value)))


def as_frame(rows):
    return pd.DataFrame([to_jsonable(row) for row in rows])

## 1. Choose one query and runtime settings

Use `RUN_LIVE_ANSWER_SYNTHESIS = True` only when you want to call the configured answer model. Query extraction will also use the query-extraction LLM automatically when `QUERY_EXTRACTION_OPENAI_API_KEY` and `QUERY_EXTRACTION_OPENAI_MODEL` are configured.

In [ ]:
QUERY = "I have asthma and a migraine. Can I take aspirin or ibuprofen while breastfeeding?"

# Retrieval settings mirror the app-level question flow defaults.
OPENFDA_LIMIT = load_query_answer_parameters().default_openfda_limit
RXNORM_DEPTH = 2
MAX_RXNORM_EDGES = 400

# Keep this false for a cheap dry run. Flip it when you want the LLM answer call.
RUN_LIVE_ANSWER_SYNTHESIS = False

print(f"Query: {QUERY}")
print(f"OpenFDA label limit: {OPENFDA_LIMIT}")
print("Query extraction LLM configured:", bool(os.getenv("QUERY_EXTRACTION_OPENAI_API_KEY") and os.getenv("QUERY_EXTRACTION_OPENAI_MODEL")))
print("Answer synthesis LLM configured:", bool(os.getenv("ANSWER_SYNTHESIS_OPENAI_API_KEY") and os.getenv("ANSWER_SYNTHESIS_OPENAI_MODEL")))

## 2. Instantiate the backend services

This uses the same request-time data stack as the FastAPI app: RxNorm parquet files plus OpenFDA live lookup with a local cache.

In [ ]:
rxnorm_store = RxNormParquetStore()
openfda_store = OpenFDALabelStore(use_cache=True, allow_live=True)
builder = DossierBuilder(rxnorm_store=rxnorm_store, openfda_store=openfda_store)

extractor = HybridQueryExtractor()
understanding_service = QueryUnderstandingService(builder=builder, extractor=extractor)
synthesizer = EvidenceAnswerSynthesizer()
query_answer_service = QueryAnswerService(
    builder=builder,
    understanding_service=understanding_service,
    synthesizer=synthesizer,
)

print("Services ready.")

## 3. Deterministic extraction only

This cell runs the rule-based extractor directly. It is the first symbolic pass before any optional LLM revision.

In [ ]:
deterministic_extraction = extractor._extract_deterministic(QUERY)
show_json(deterministic_extraction.state)
as_frame(deterministic_extraction.mentions)

## 4. Hybrid extraction

`extractor.extract(...)` runs deterministic extraction first and then lets the query-extraction LLM revise it when configured. Inspect `mode`, `warnings`, and `errors` to see what happened.

In [ ]:
hybrid_extraction = extractor.extract(QUERY)

print("Extraction mode:", hybrid_extraction.mode)
print("Warnings:", hybrid_extraction.warnings)
print("Errors:", hybrid_extraction.errors)
show_json(hybrid_extraction.state)
as_frame(hybrid_extraction.mentions)

## 5. Resolve extracted drug mentions through RxNorm

The app resolves each extracted drug-like mention to RxNorm candidates. This cell exposes those candidates before the full `QueryUnderstandingResponse` is built.

In [ ]:
resolved_mentions = understanding_service._resolve_extraction(
    QUERY,
    hybrid_extraction,
)

rows = []
for mention in resolved_mentions:
    rows.append(
        {
            "mention": mention.text,
            "role": mention.role,
            "selected_rxcui": mention.selected_concept.rxcui if mention.selected_concept else None,
            "selected_name": mention.selected_concept.name if mention.selected_concept else None,
            "selected_tty": mention.selected_concept.tty if mention.selected_concept else None,
            "candidate_count": len(mention.candidates),
            "top_match_type": mention.candidates[0].match_type if mention.candidates else None,
            "top_score": mention.candidates[0].score if mention.candidates else None,
        }
    )

pd.DataFrame(rows)

## 6. Select the primary medication

This is the concept that drives the full dossier retrieval in V1.

In [ ]:
primary_mention = understanding_service._select_primary(
    resolved_mentions,
    hybrid_extraction.state.primary_drug,
)

show_json(primary_mention)

In [ ]:
show_json(primary_mention.selected_concept)

## 7. Build the primary drug dossier

This calls RxNorm for the local network and OpenFDA for label evidence. The OpenFDA store uses cache first and live lookup when needed.

In [ ]:
if primary_mention and primary_mention.selected_concept:
    primary_dossier = builder.build(
        primary_mention.selected_concept.name,
        depth=RXNORM_DEPTH,
        max_edges=MAX_RXNORM_EDGES,
        include_openfda=True,
        openfda_limit=OPENFDA_LIMIT,
    )
else:
    primary_dossier = None

show_json(primary_dossier.resolved_drug if primary_dossier else None)
print("Notes:", primary_dossier.notes if primary_dossier else [])

## 8. Inspect RxNorm candidates and neighborhood

In [ ]:
if primary_dossier:
    display(as_frame(primary_dossier.resolution_candidates).head(10))
    neighborhood = primary_dossier.rxnorm_neighborhood
    print("Depth:", neighborhood.depth)
    print("Nodes:", len(neighborhood.nodes))
    print("Edges:", len(neighborhood.edges))
    print("Truncated:", neighborhood.truncated)
    display(as_frame(neighborhood.edges).head(15))

## 9. Inspect OpenFDA label records

These are the source records that later become the Drug Labels sources in the UI.

In [ ]:
label_evidence = primary_dossier.label_evidence if primary_dossier else None

if label_evidence:
    print("Retrieval mode:", label_evidence.retrieval_mode)
    print("Labels found:", label_evidence.labels_found)
    print("Label limit:", label_evidence.label_limit)
    display(
        as_frame(label_evidence.label_records)[
            [
                "source_id",
                "brand_names",
                "generic_names",
                "manufacturer_names",
                "routes",
                "product_types",
                "rxcuis",
            ]
        ]
    )
else:
    print("No label evidence.")

## 10. Inspect normalized label sections

The OpenFDA API returns many raw fields. The dossier layer keeps a prioritized set of label sections and normalizes aliases like `warnings_and_precautions` into `warnings`.

In [ ]:
if label_evidence:
    section_rows = []
    for section_name, entries in label_evidence.sections.items():
        for entry in entries:
            section_rows.append(
                {
                    "section": section_name,
                    "source_id": entry.source_id,
                    "effective_time": entry.effective_time,
                    "text_chars": len(entry.text),
                    "text_preview": entry.text[:240].replace("\n", " "),
                }
            )

    sections_df = pd.DataFrame(section_rows)
    display(sections_df.groupby("section").size().rename("entries").reset_index())
    display(sections_df.head(12))
else:
    print("No label sections.")

## 11. Run the public query-understanding service

This is the object returned by `POST /query-understanding`. It combines extraction, RxNorm resolution, and the primary dossier.

In [ ]:
understanding = understanding_service.understand(
    QUERY,
    openfda_limit=OPENFDA_LIMIT,
)

print("Extraction mode:", understanding.extraction_mode)
print("Warnings:", understanding.warnings)
print("Errors:", understanding.errors)
show_json(understanding.state)

## 12. Build the compact evidence packet for answer synthesis

This is the grounded packet sent to the answer-synthesis LLM. It includes extracted state, the resolved primary concept, a compact RxNorm relationship summary, source metadata, prioritized label sections, and retrieval notes.

In [ ]:
evidence_packet = synthesizer.build_evidence_packet(understanding)

print("Top-level keys:", list(evidence_packet.keys()))
print("Label sources:", len(evidence_packet.get("label_sources", [])))
print("Label sections:", len(evidence_packet.get("label_sections", [])))
show_json(evidence_packet)

## 13. Inspect allowed citation targets

Generated source citations are filtered against this set. If the model cites anything else, it is dropped or retried by the synthesis layer.

In [ ]:
allowed_citations = synthesizer.allowed_citations(evidence_packet)
pd.DataFrame(
    sorted(allowed_citations),
    columns=["source_id", "section"],
)

## 14. Inspect the answer prompt messages without calling the model

This shows the exact message payload shape after placeholders are filled. The evidence packet can be large, so this cell displays lengths plus a preview.

In [ ]:
prompt_config = synthesizer._load_prompt()
messages = synthesizer._format_messages(
    prompt_config.get("messages", []),
    query=QUERY,
    evidence_packet=json.dumps(evidence_packet, indent=2),
)

prompt_rows = [
    {
        "role": message["role"],
        "chars": len(message["content"]),
        "preview": message["content"][:600],
    }
    for message in messages
]
display(pd.DataFrame(prompt_rows))
display(Markdown(f"```json\n{json.dumps(prompt_config, indent=2)}\n```"))

## 15. Optional: call live answer synthesis

This calls the answer-synthesis model only when `RUN_LIVE_ANSWER_SYNTHESIS` is true and answer synthesis env vars are configured.

In [ ]:
if RUN_LIVE_ANSWER_SYNTHESIS:
    synthesis_result = synthesizer.synthesize(QUERY, understanding)
else:
    synthesis_result = None
    print("Skipped live answer synthesis. Set RUN_LIVE_ANSWER_SYNTHESIS = True to call it.")

if synthesis_result:
    print("Warnings:", synthesis_result.warnings)
    print("Errors:", synthesis_result.errors)
    show_json(synthesis_result.answer)

## 16. Parse a mock answer without an LLM call

This is useful when you want to inspect citation filtering and answer shape without spending tokens. The fake answer cites the first allowed source-section pair if one exists.

In [ ]:
first_citation = next(iter(sorted(allowed_citations)), None)
mock_raw_answer = {
    "summary": "Mock educational summary for backend inspection.",
    "bullets": [
        {
            "text": "This bullet is linked to a retrieved label section when one is available.",
            "citations": [
                {"source_id": first_citation[0], "section": first_citation[1]}
            ]
            if first_citation
            else [],
        }
    ],
    "limitations": ["Mock limitation from the model output."],
}

mock_answer = synthesizer.parse_answer_data(mock_raw_answer, allowed_citations)
show_json(mock_answer)

## 17. Deterministic evidence coverage

Coverage compares the extracted symbolic state against the retrieved primary-drug evidence. It is deterministic transparency metadata, not another LLM judge.

In [ ]:
coverage = build_evidence_coverage(understanding)

print("Summary counts:", coverage.summary_counts)
as_frame(coverage.items)

## 18. Add deterministic coverage limitations to an answer

The app keeps the LLM answer but appends deterministic limitations when coverage shows important gaps.

In [ ]:
answer_for_guardrail_demo = (
    synthesis_result.answer
    if synthesis_result and synthesis_result.answer
    else mock_answer
)

guardrailed_answer = add_coverage_limitations(
    answer_for_guardrail_demo,
    coverage,
    understanding,
)

show_json(guardrailed_answer)

In [ ]:
display(synthesis_result.answer.limitations)
display(guardrailed_answer.limitations)

## 19. Optional: run the full query-answer service

This mirrors `POST /query-answer`: understanding, synthesis, coverage, deterministic limitations, warnings, and errors. It can call the answer-synthesis LLM, so it is guarded by the same toggle.

In [ ]:
if RUN_LIVE_ANSWER_SYNTHESIS:
    full_response = query_answer_service.answer(
        QUERY,
        openfda_limit=OPENFDA_LIMIT,
    )
    show_json(full_response)
else:
    print("Skipped full query-answer service call because it may call answer synthesis.")

## 20. What to tweak while debugging

- Change `QUERY` and rerun cells from section 3 onward.
- Increase or decrease `OPENFDA_LIMIT` to inspect how many label records reach the evidence packet.
- Compare deterministic extraction vs. hybrid extraction to see what the query-extraction LLM corrected.
- Inspect `evidence_packet["label_sections"]` when a generated response feels under-grounded.
- Inspect `coverage.items` when limitations are added or missing.